## Quickstart
### Explore in the browser
The Explorer site is the fastest way to browse Overture data and inspect the schema. No installation, no account. Click any feature to view its properties. Download visible data as GeoJSON.

### Download by area of interest
Install Overture's Python client and download building footprints for a specific area:

In [2]:
# Install the library first
# !pip install overturemaps

# Run the download command
!overturemaps download \
  --bbox=-71.068,42.353,-71.058,42.363 \
  -f geojson \
  --type=building \
  -o boston_buildings.geojson

State saved to boston_buildings.geojson.state


The tool reads directly from Overture's cloud-hosted GeoParquet and transfers only the data inside your bounding box.

### Query with DuckDB
DuckDB lets you query Overture's GeoParquet files with SQL. Install DuckDB, then:

In [6]:
LOAD spatial;
LOAD httpfs;

SET s3_region = 'us-west-2';

COPY (
  SELECT
    id,
    names.primary AS name,
    categories.primary AS category,
    -- Note: Overture addresses are often lists; [1] accesses the first element
    addresses[1].freeform AS address,
    geometry
  FROM read_parquet(
    's3://overturemaps-us-west-2/release/2024-03-12.0/theme=places/type=place/*', 
    hive_partitioning=1
  )
  WHERE
    names.primary ILIKE '%wawa%'
    -- Filter using the spatial columns provided by Overture for performance
    AND bbox.xmin > -76.5 
    AND bbox.xmax < -74.5
    AND bbox.ymin > 39.5 
    AND bbox.ymax < 40.5
) TO 'wawa_stores.geojson' WITH (FORMAT 'GeoJSON');

SyntaxError: invalid syntax (2486777581.py, line 1)

This extracts Wawa convenience stores in the Philadelphia area and saves them as GeoJSON.

### Get the latest release
Overture publishes a STAC catalog that always points to the latest release. Query it with DuckDB instead of hardcoding release paths:

In [7]:
SET VARIABLE latest = (
    SELECT latest FROM 'https://stac.overturemaps.org/catalog.json'
);

SyntaxError: invalid syntax (886226411.py, line 1)